In [ ]:
import json
import random
from openai import OpenAI
from tqdm import tqdm
import random
from datasets import Dataset
import os
import pickle
from dotenv import load_dotenv
from config import Config


d:\Kuliah\SKRIPSI\code\playground\skripsi-playground-venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PICKLE_BACKUP_PATH = Config.PICKLE_PATH
OUTPUT_PATH = Config.GOLDEN_DATASET_PATH


In [11]:
def generate_multi_context_qa(client, contexts, n_questions=2):
    combined_context = "\n\n---\n\n".join(contexts)
    prompt = f"""
    Anda bertugas membuat dataset evaluasi berkualitas tinggi untuk sistem RAG medis.
    Gunakan HANYA informasi yang terdapat pada bagian "Konteks" di bawah ini.

    Instruksi:
    1. Buat {n_questions} pertanyaan multi-hop.
    2. Pertanyaan WAJIB menghubungkan minimal 2 bagian informasi yang berbeda dari konteks yang sama.
    3. Fokus pada narasi, intervensi, atau edukasi.
    4. DILARANG menanyakan diagnosis medis, data antropometri (BB/TB), atau angka laboratorium secara langsung.

    Keluaran WAJIB dalam format JSON:
    [
        {"question": "...", "answer": "..."}
    ]

    Konteks:
    {combined_context}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.3,
        )
        content = json.loads(response.choices[0].message.content)
        if isinstance(content, dict):
            for key in ['questions', 'data', 'qa_pairs']:
                if key in content: return content[key]
            return [content]
        return content
    except Exception as e:
        print(f"Error generating QA: {e}")
        return []


In [12]:
def build_grouped_docs(documents):
    grouped = {}
    for doc in documents:
        source = doc.metadata.get('source', 'Unknown')
        if source not in grouped:
            grouped[source] = []
        grouped[source].append(doc)
    return grouped

def build_dataset(grouped_docs, client, max_samples=50):
    dataset = []
    sources = list(grouped_docs.keys())

    pbar = tqdm(total=max_samples, desc="Generating Valid Multi-Hop Data")
    while len(dataset) < max_samples:
        source = random.choice(sources)
        docs_in_source = grouped_docs[source]

        if len(docs_in_source) < 2: continue

        # Pick 2-3 chunks from the SAME source
        num_chunks = min(3, len(docs_in_source))
        selected_docs = random.sample(docs_in_source, num_chunks)
        selected_texts = [d.page_content for d in selected_docs]

        qa_pairs = generate_multi_context_qa(client, selected_texts)

        if not isinstance(qa_pairs, list): qa_pairs = [qa_pairs]

        for qa in qa_pairs:
            if len(dataset) >= max_samples: break
            if not qa or 'question' not in qa: continue

            dataset.append({
                "question": qa["question"],
                "ground_truth": qa.get("answer") or qa.get("ground_truth"),
                "gold_contexts": selected_texts,
                "chunk_metadata": [d.metadata for d in selected_docs]
            })
            pbar.update(1)

    pbar.close()
    return dataset


In [7]:
def save_dataset(dataset, path=OUTPUT_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(dataset)} samples to {path}")


In [ ]:
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

PICKLE_PATH = "knowledge-base/2026_04_29__1545_medical_documents_enriched.pkl"
OUTPUT_PATH = "nutrition_qa_golden_dataset.json"

print(f"Loading documents from {PICKLE_PATH}...")
with open(PICKLE_PATH, 'rb') as f:
    documents = pickle.load(f)

print("Grouping documents by source...")
grouped = build_grouped_docs(documents)

print("Starting generation...")
final_dataset = build_dataset(grouped, client, max_samples=50)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=2, ensure_ascii=False)

print(f"Saved {len(final_dataset)} samples to {OUTPUT_PATH}")


  1%|▏         | 2/150 [00:17<20:43,  8.40s/it]

Format output tidak dikenali


 14%|█▍        | 21/150 [02:39<17:39,  8.21s/it]

Format output tidak dikenali


 17%|█▋        | 25/150 [03:07<14:48,  7.11s/it]

...

Format output tidak dikenali


 35%|███▍      | 52/150 [05:59<10:13,  6.26s/it]

Format output tidak dikenali


 41%|████▏     | 62/150 [07:11<10:12,  6.96s/it]

Saved 150 samples to nutrition_qa_golden_dataset.json
